# mHC inside DeepSeek-V4: a widened residual stream you can actually train
### Notebook 2 of 9 on DeepSeek-V4 (technical report [arXiv:2606.19348](https://arxiv.org/abs/2606.19348), §2.2 and §3.3-3.4.2)

Every Transformer block in V4 sits between two small mixing layers that replace the plain residual add. The residual state is widened from `d` to `n_hc × d` numbers per token (`n_hc = 4`), and the matrix that mixes the four streams is constrained to be **doubly stochastic** with the Sinkhorn-Knopp algorithm (20 iterations), which keeps the composite mixing from amplifying or erasing the signal across 43 or 61 layers. The method is the one introduced in [Xie et al., 2026](https://arxiv.org/abs/2512.24880), and the report points there for the engineering details.

**Headline numbers from the V4 report.** `n_hc = 4`, `t_max = 20`, in both V4-Flash and V4-Pro (§4.2.1); the wall-time overhead of mHC is constrained to **6.7 % of the overlapped 1F1B pipeline stage** (§3.4.2).

**This notebook is deliberately short.** The derivation, the properties (spectral norm, closure, Birkhoff decomposition), the Sinkhorn-Knopp convergence study and a small training experiment are in our standalone notebook [`../models/mhc/mhc_from_scratch.ipynb`](../models/mhc/mhc_from_scratch.ipynb). Here we do only what is specific to **V4**: map the notation, count the cost at V4 sizes, and reproduce a systems detail the report calls out, the deterministic reduction of mHC's 24-wide matrix multiply.

| section | question | number it carries |
|---|---|---|
| 1 | How does V4's notation map onto the mHC paper's? | `n_hc² + 2·n_hc = 24` coefficients |
| 2 | What does mHC add at V4-Pro size? | parameters, pipeline traffic |
| 3 | Does the doubly-stochastic guarantee hold across V4's depth? | composite gain over 122 mixing layers |
| 4 | Why does a 24-wide matmul need a special deterministic kernel? | bitwise differences across reduction orders |

<p align="center"><img src="images/fig2_architecture.png" width="420"><br><sub>Fig. 2 (report). Notice the two 'Residual Mixing' / 'Pre-Block Mixing' / 'Post-Block Mixing' groups inside each Transformer block: the attention sublayer (CSA/HCA) and the MoE sublayer each get their own set of mixing matrices, so a 61-layer model has 122 mHC sites.</sub></p>

In [1]:
import math, time, itertools
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
plt.rcParams.update({"figure.dpi": 105, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25, "font.size": 9})
CHECKS = []
def check(name, ok, detail=""): CHECKS.append((name, bool(ok), detail)); print(("PASS " if ok else "FAIL ") + name + (f"   [{detail}]" if detail else ""))

## Notation and a layer in code

V4 writes the update as `X_{l+1} = B_l X_l + C_l F_l(A_l X_l)` with `X_l ∈ R^{n_hc×d}`, an input mapping `A_l ∈ R^{1×n_hc}`, a residual mapping `B_l ∈ R^{n_hc×n_hc}` and an output mapping `C_l ∈ R^{n_hc×1}` (Eq. 1). In the mHC paper the same three objects are called `H_pre`, `H_res` and `H_post`. The constraints are (Eqs. 2, 6, 7): `B_l` doubly stochastic (Sinkhorn-Knopp on `exp(B̃_l)`), `A_l = σ(Ã_l)` and `C_l = 2σ(C̃_l)`, so both are non-negative and bounded. The raw values are generated from the current state, `X̂_l = RMSNorm(vec(X_l))`, as `α · (X̂_l W) + S` (a dynamic part scaled by a learnable gate `α` initialized small, plus a static bias `S`).

The output of that projection is `n_hc + n_hc + n_hc² = 24` numbers per token for `n_hc = 4`, which is why §3.3 describes mHC's matmul as having "an output dimension of only 24".

In [2]:
def sinkhorn_knopp(logits, t_max=20):
    M = logits.exp()
    for _ in range(t_max):
        M = M / M.sum(-2, keepdim=True); M = M / M.sum(-1, keepdim=True)
    return M
def rms(x, eps=1e-6): return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps)

class MHC(nn.Module):
    """V4 notation: X_{l+1} = B X + C F(A X).  X: (T, n_hc, d)."""
    def __init__(s, d, n_hc=4, t_max=20, alpha0=0.01, kappa=10.0):
        super().__init__(); s.n, s.t_max = n_hc, t_max
        s.W = nn.Parameter(torch.randn(n_hc * d, n_hc * n_hc + 2 * n_hc) * 0.02)                # one projection producing all 24 raw values: [B~ (16) | A~ (4) | C~ (4)]
        s.alpha = nn.Parameter(torch.full((3,), alpha0))                                          # gating factors for res, pre, post
        s.S_res = nn.Parameter(kappa * torch.eye(n_hc)); s.S_pre = nn.Parameter(torch.full((n_hc,), -math.log(n_hc - 1))); s.S_post = nn.Parameter(torch.zeros(n_hc))
    def coeffs(s, X):
        T, n, d = X.shape; raw = rms(X.reshape(T, n * d)) @ s.W                                   # (T, 24)
        Bt = s.alpha[0] * raw[:, :n * n].view(T, n, n) + s.S_res
        At = s.alpha[1] * raw[:, n * n:n * n + n] + s.S_pre; Ct = s.alpha[2] * raw[:, n * n + n:] + s.S_post
        return torch.sigmoid(At), sinkhorn_knopp(Bt, s.t_max), 2 * torch.sigmoid(Ct)               # A (T,n), B (T,n,n), C (T,n)
    def forward(s, X, F_):
        A, B, C = s.coeffs(X)
        return torch.einsum("tnm,tmd->tnd", B, X) + C[:, :, None] * F_(torch.einsum("tn,tnd->td", A, X))[:, None, :]

d_ = 32; layer = MHC(d_); X = torch.randn(10, 4, d_); A, B, C = layer.coeffs(X)
print("coefficients per token:", layer.W.shape[1], " = n_hc^2 + 2 n_hc =", 4 * 4 + 2 * 4)
check("n_hc^2 + 2 n_hc = 24 raw coefficients per token for n_hc = 4 (the 'output dimension of only 24' in report §3.3)", layer.W.shape[1] == 24)
check("A in (0,1), C in (0,2), B doubly stochastic (rows 1 exactly, columns within 1e-3)", bool((A > 0).all() and (A < 1).all() and (C > 0).all() and (C < 2).all() and (B.sum(-1) - 1).abs().max() < 1e-5 and (B.sum(-2) - 1).abs().max() < 1e-3))
Xo = layer(X, lambda h: 0.1 * h); print("output shape", tuple(Xo.shape))
check("a fresh layer with F = 0.1*identity acts like x + F(x) on the stream average (A=1/n, C=1, B~I)", torch.allclose(Xo.mean(1), X.mean(1) + 0.1 * X.mean(1), atol=5e-3), f"max err {(Xo.mean(1) - 1.1 * X.mean(1)).abs().max():.1e}")

coefficients per token: 24  = n_hc^2 + 2 n_hc = 24
PASS n_hc^2 + 2 n_hc = 24 raw coefficients per token for n_hc = 4 (the 'output dimension of only 24' in report §3.3)
PASS A in (0,1), C in (0,2), B doubly stochastic (rows 1 exactly, columns within 1e-3)
output shape (10, 4, 32)
PASS a fresh layer with F = 0.1*identity acts like x + F(x) on the stream average (A=1/n, C=1, B~I)   [max err 2.2e-04]


The initialization (`A = 1/n`, `C = 1`, `B ≈ I`) is our choice: the report gives the gate scale as "small" but not the biases (our standalone notebook uses the same). It makes a freshly initialized mHC layer behave like an ordinary residual layer on the stream average, which the last check above confirms.

Next, what does this cost at V4's size?

## The price at V4-Pro size

Two costs follow directly from widening the stream, and the report gives the configuration needed to count them (V4-Pro: `d = 7168`, 61 layers, `n_hc = 4`).

**Parameters.** Per mHC site, the projection `W ∈ R^{n_hc d × 24}` has `4 · 7168 · 24 = 688,128` parameters, plus `n² + 2n = 24` biases and 3 gates. A block has two sites (attention and MoE), so 61 blocks give 122 sites.

**Pipeline traffic.** With pipeline parallelism, the activation sent between stages is the residual state: `d` numbers per token for a plain residual, `n_hc · d` for mHC, i.e. `4×` the communication volume. This is why the report extends its pipeline schedule (§3.4.2) rather than leaving it as is.

In [3]:
d, n_hc, L = 7168, 4, 61
per_site = n_hc * d * (n_hc * n_hc + 2 * n_hc) + (n_hc * n_hc + 2 * n_hc) + 3; sites = 2 * L; total = per_site * sites
print(f"parameters per mHC site: {per_site:,};  sites: {sites};  total {total / 1e6:.1f}M = {100 * total / 1.6e12:.4f}% of 1.6T parameters")
fl = 2 * n_hc * d * (n_hc * n_hc + 2 * n_hc) * sites
print(f"coefficient matmul FLOPs per token (forward): {fl / 1e6:.0f}M  vs ~{2 * 49e9 / 1e9:.0f}G for the 49B activated parameters = {100 * fl / (2 * 49e9):.3f}%")
print(f"pipeline-boundary activation per token (bf16): residual {d * 2 / 1024:.1f} KiB  ->  mHC {n_hc * d * 2 / 1024:.1f} KiB  ({n_hc}x)")
check("mHC coefficient parameters are below 0.01% of V4-Pro's 1.6T", total / 1.6e12 < 1e-4, f"{100 * total / 1.6e12:.4f}%")
check("mHC coefficient FLOPs are below 0.5% of the activated-parameter FLOPs", fl / (2 * 49e9) < 5e-3, f"{100 * fl / (2 * 49e9):.3f}%")

parameters per mHC site: 688,155;  sites: 122;  total 84.0M = 0.0052% of 1.6T parameters
coefficient matmul FLOPs per token (forward): 168M  vs ~98G for the 49B activated parameters = 0.171%
pipeline-boundary activation per token (bf16): residual 14.0 KiB  ->  mHC 56.0 KiB  (4x)
PASS mHC coefficient parameters are below 0.01% of V4-Pro's 1.6T   [0.0052%]
PASS mHC coefficient FLOPs are below 0.5% of the activated-parameter FLOPs   [0.171%]


**What we found.** By parameters (about 0.005 %) and FLOPs (about 0.17 % by our count) mHC is nearly free at V4-Pro size, and the report's *measured* overhead is 6.7 % of the pipeline stage: roughly forty times the FLOP share. The gap is memory traffic and communication, not arithmetic, which is what the report's fused kernels, selective recomputation ("we recompute most hidden states between layers and all normalized layer inputs, while avoiding recomputation of compute-intensive operations") and modified DualPipe schedule address. The quantitative treatment of those three is in the standalone notebook (Table 2 of the mHC paper reproduced, and the recomputation block size). We did not write any GPU kernels.

Does the stability guarantee survive V4's depth?

## Stability across 122 mixing layers

Because doubly stochastic matrices are closed under multiplication, the composite mapping over any number of layers is doubly stochastic, and its *forward* gain (max row sum) and *backward* gain (max column sum) are exactly 1. In practice Sinkhorn-Knopp is stopped after 20 iterations, so each matrix is only approximately doubly stochastic and the errors compound. We measure the composite gain over V4-Pro's 122 sites, for the logit scales a trained `B̃` might have (standard deviation 0.5 to 3; sharper means closer to a permutation, the slow case for Sinkhorn-Knopp), and compare 20 iterations with 100.

In [4]:
def composite_gain(t_max, s, L=122, seeds=100, n=4):
    out = []
    for sd in range(seeds):
        g = torch.Generator().manual_seed(sd); H = sinkhorn_knopp(torch.randn(L, n, n, generator=g) * s, t_max); P = torch.eye(n)
        for l in range(L): P = H[l] @ P
        out.append((P.abs().sum(1).amax().item(), P.abs().sum(0).amax().item()))
    return torch.tensor(out)
print(f"{'logit std':>9s} {'t_max':>6s} {'fwd gain (median/max)':>24s} {'bwd gain (median/max)':>24s}")
res = {}
for s in [0.5, 1.0, 2.0, 3.0]:
    for t in [20, 100]:
        r = composite_gain(t, s); res[(s, t)] = r; print(f"{s:9.1f} {t:6d} {r[:, 0].median():12.4f} /{r[:, 0].max():8.4f} {r[:, 1].median():14.4f} /{r[:, 1].max():8.4f}")
check("with logit std <= 1 and t_max = 20, composite gains over 122 layers stay within 1.05", all(res[(s, 20)].max() < 1.05 for s in (0.5, 1.0)), f"{max(res[(s, 20)].max().item() for s in (0.5, 1.0)):.4f}")
check("100 iterations always tighten the worst-case gain relative to 20 (same matrices' logits)", all(res[(s, 100)].max() <= res[(s, 20)].max() + 1e-6 for s in (0.5, 1.0, 2.0, 3.0)))
check("sharper logits (std 3) at t_max = 20 give a visibly larger worst-case gain than std 0.5", res[(3.0, 20)].max() > res[(0.5, 20)].max())

logit std  t_max    fwd gain (median/max)    bwd gain (median/max)


      0.5     20       1.0000 /  1.0000         1.0000 /  1.0000


      0.5    100       1.0000 /  1.0000         1.0000 /  1.0000
      1.0     20       1.0000 /  1.0000         1.0000 /  1.0000


      1.0    100       1.0000 /  1.0000         1.0000 /  1.0000


      2.0     20       1.0000 /  1.0000         1.0002 /  1.0071


      2.0    100       1.0000 /  1.0000         1.0000 /  1.0000


      3.0     20       1.0000 /  1.0000         1.0041 /  1.0254


      3.0    100       1.0000 /  1.0000         1.0000 /  1.0017
PASS with logit std <= 1 and t_max = 20, composite gains over 122 layers stay within 1.05   [1.0000]
PASS 100 iterations always tighten the worst-case gain relative to 20 (same matrices' logits)
PASS sharper logits (std 3) at t_max = 20 give a visibly larger worst-case gain than std 0.5


**What we found.** For well-conditioned logits the composite gain over 122 layers is within a few percent of 1 even after only 20 iterations; for sharper matrices the same 20 iterations leave visibly more drift, and more iterations tighten it. These are random draws, not trained values, so they bound the *mechanism*, not V4's actual gains (the mHC paper's Fig. 7 measures real ones at 27B). Neither paper ablates `t_max`, so the choice of 20 is not something we can defend or criticize.

One more V4-specific point, which is about bits, not about stability.

## Why a 24-wide matmul needs a custom deterministic kernel

V4 targets **bitwise reproducibility**: the same input must give the same bits, run after run and between training and inference (§3.3). Floating-point addition is not associative, so any kernel whose accumulation order can vary is a problem. The report singles out mHC: the coefficient projection multiplies a `[tokens, n_hc·d]` matrix by `[n_hc·d, 24]`. The output is tiny (24 columns), so with few tokens there are not enough output tiles to occupy the GPU, and the standard remedy is **split-k**: cut the long reduction dimension into `S` pieces, compute partial sums in parallel, then add them. A naive implementation adds the partials with atomics in whatever order the hardware completes them, which is non-deterministic. The report's fix: "we output each split part separately and perform a deterministic reduction in a subsequent kernel."

We reproduce the effect in float32 on CPU at V4-Pro's reduction length (`n_hc · d = 28,672`), simulating the atomic order by shuffling the partials.

In [5]:
torch.manual_seed(0); K = 4 * 7168; x = torch.randn(1, K) * 3; W = torch.randn(K, 24) * 0.02            # one token, V4-Pro reduction length, 24 outputs
def partials(S): return torch.stack([x[:, i * (K // S):(i + 1) * (K // S)] @ W[i * (K // S):(i + 1) * (K // S)] for i in range(S)])   # (S,1,24) in float32
S = 16; P = partials(S)
def reduce_in_order(order):
    acc = torch.zeros(1, 24)
    for i in order: acc = acc + P[i]                                                                   # float32 accumulation in a given order
    return acc
ref = reduce_in_order(range(S)); diffs = []
for _ in range(200): diffs.append((reduce_in_order(torch.randperm(S).tolist()) - ref).abs().max().item())
n_bit_diff = sum(d_ > 0 for d_ in diffs)
print(f"naive 'atomic-like' reduction: {n_bit_diff}/200 random completion orders give a different float32 result; largest difference {max(diffs):.2e}")
fixed = [reduce_in_order(range(S)) for _ in range(200)]
check("fixed-order reduction of the separately written partials is bit-identical on every run", all(torch.equal(f, ref) for f in fixed))
check("a random accumulation order changes the float32 bits in most trials (this is the non-determinism the report removes)", n_bit_diff > 100, f"{n_bit_diff}/200")
r1, r2 = reduce_in_order(range(16)), (lambda Pp: sum(Pp[i] for i in range(8)))(torch.stack([x[:, i * (K // 8):(i + 1) * (K // 8)] @ W[i * (K // 8):(i + 1) * (K // 8)] for i in range(8)]))
print(f"S=16 vs S=8 (fixed order each): max difference {(r1 - r2).abs().max():.2e}   (different split counts are different numerics)")
check("changing the number of splits changes the result, so determinism ALSO requires the split count not to depend on the batch size", (r1 - r2).abs().max() > 0)

naive 'atomic-like' reduction: 200/200 random completion orders give a different float32 result; largest difference 5.72e-06
PASS fixed-order reduction of the separately written partials is bit-identical on every run
PASS a random accumulation order changes the float32 bits in most trials (this is the non-determinism the report removes)   [200/200]
S=16 vs S=8 (fixed order each): max difference 1.05e-05   (different split counts are different numerics)
PASS changing the number of splits changes the result, so determinism ALSO requires the split count not to depend on the batch size


**What we found.** With a random accumulation order the float32 result changes in most trials, at a relative size around `1e-7`, which is the rounding noise one expects; with the partials written separately and reduced in a fixed order it is identical every time. The last check shows the second half of the requirement: even a deterministic reduction gives different bits for different split counts, so **batch invariance** (the same token giving the same bits regardless of how many other tokens share its batch) additionally needs the split count to be a fixed function of the shape, not of the batch. Notebook 9 builds that argument out for attention and for the other matrix multiplies.

## Closing thoughts

V4 inherits mHC from the mHC paper and adds nothing to the method; what the V4 report contributes is the deployment evidence (`n_hc = 4`, `t_max = 20`, two mixing sites per block, 6.7 % pipeline-stage overhead) and the determinism work around it. In this notebook the numbers that matter are small ones: 24 coefficients per token, about 0.005 % of the parameters, a `4×` increase in pipeline-boundary traffic, and a composite gain within a few percent of 1 over 122 layers for well-conditioned logits.

**What is still open.** Neither report ablates the number of Sinkhorn-Knopp iterations or reports per-component overheads, and our gain measurements use random logits rather than trained ones. Whether another manifold would beat the Birkhoff polytope is left as future work by the mHC authors.

**Next.** Notebook 3 covers the optimizer that trains all of this, Muon with a hybrid Newton-Schulz orthogonalization.

Credits: mHC is from Xie et al. (DeepSeek-AI); Hyper-Connections from Zhu et al.; the projection is the 1967 algorithm of Sinkhorn and Knopp.

In [6]:
print(f"{'#':>2s}  {'status':6s}  check")
for i, (n_, ok, d_) in enumerate(CHECKS, 1): print(f"{i:2d}  {'PASS' if ok else 'FAIL':6s}  {n_}")
failed = [n_ for n_, ok, _ in CHECKS if not ok]; print(f"\n{len(CHECKS) - len(failed)} passed, {len(failed)} failed")
assert not failed, failed

 #  status  check
 1  PASS    n_hc^2 + 2 n_hc = 24 raw coefficients per token for n_hc = 4 (the 'output dimension of only 24' in report §3.3)
 2  PASS    A in (0,1), C in (0,2), B doubly stochastic (rows 1 exactly, columns within 1e-3)
 3  PASS    a fresh layer with F = 0.1*identity acts like x + F(x) on the stream average (A=1/n, C=1, B~I)
 4  PASS    mHC coefficient parameters are below 0.01% of V4-Pro's 1.6T
 5  PASS    mHC coefficient FLOPs are below 0.5% of the activated-parameter FLOPs
 6  PASS    with logit std <= 1 and t_max = 20, composite gains over 122 layers stay within 1.05
 7  PASS    100 iterations always tighten the worst-case gain relative to 20 (same matrices' logits)
 8  PASS    sharper logits (std 3) at t_max = 20 give a visibly larger worst-case gain than std 0.5
 9  PASS    fixed-order reduction of the separately written partials is bit-identical on every run
10  PASS    a random accumulation order changes the float32 bits in most trials (this is the non-determini